In [31]:
import pandas as pd
customers = pd.read_csv("../data/raw/customers.csv")
customers.head()

,customer_id,customer_type,industry_segment,city,state,pincode,region,branch_id,credit_limit,current_balance,payment_terms,customer_since,last_purchase_date,total_purchase_value,customer_rating
0,C0001,Retail,Construction,Kolkata,West Bengal,991035,East,KOL001,45896,43284,Advance,2024-04-10,2024-04-25,3892818,4
1,C0002,Fleet Owner,Manufacturing,Hyderabad,Telangana,464589,South,HYD001,531506,426735,Advance,2022-01-19,2024-12-04,3168730,2
2,C0003,Government,Logistics,Surat,Gujarat,202475,West,DEL001,1306933,733495,Net 30,2017-02-14,2019-06-11,3584708,2
3,C0004,Fleet Owner,Logistics,Chennai,Tamil Nadu,576213,South,CHN001,506336,250255,Net 30,2017-12-20,2021-12-26,3810635,3
4,C0005,Dealer,Construction,Ahmedabad,Gujarat,965661,West,AHM001,380783,183927,Net 30,2023-07-27,2024-02-17,1381374,2


In [32]:
print("=== INFO ===")
print(customers.info())

print("\n=== MISSING VALUES ===")
print(customers.isnull().sum())

print("\n=== SAMPLE DATA ===")
customers.head()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   customer_id           500 non-null    str  
 1   customer_type         500 non-null    str  
 2   industry_segment      500 non-null    str  
 3   city                  500 non-null    str  
 4   state                 500 non-null    str  
 5   pincode               500 non-null    int64
 6   region                500 non-null    str  
 7   branch_id             500 non-null    str  
 8   credit_limit          500 non-null    int64
 9   current_balance       500 non-null    int64
 10  payment_terms         500 non-null    str  
 11  customer_since        500 non-null    str  
 12  last_purchase_date    500 non-null    str  
 13  total_purchase_value  500 non-null    int64
 14  customer_rating       500 non-null    int64
dtypes: int64(5), str(10)
memory usage: 58.7 KB
None

=== MI

,customer_id,customer_type,industry_segment,city,state,pincode,region,branch_id,credit_limit,current_balance,payment_terms,customer_since,last_purchase_date,total_purchase_value,customer_rating
0,C0001,Retail,Construction,Kolkata,West Bengal,991035,East,KOL001,45896,43284,Advance,2024-04-10,2024-04-25,3892818,4
1,C0002,Fleet Owner,Manufacturing,Hyderabad,Telangana,464589,South,HYD001,531506,426735,Advance,2022-01-19,2024-12-04,3168730,2
2,C0003,Government,Logistics,Surat,Gujarat,202475,West,DEL001,1306933,733495,Net 30,2017-02-14,2019-06-11,3584708,2
3,C0004,Fleet Owner,Logistics,Chennai,Tamil Nadu,576213,South,CHN001,506336,250255,Net 30,2017-12-20,2021-12-26,3810635,3
4,C0005,Dealer,Construction,Ahmedabad,Gujarat,965661,West,AHM001,380783,183927,Net 30,2023-07-27,2024-02-17,1381374,2


In [33]:
text_cols = [
    'customer_id', 'customer_type', 'industry_segment',
    'city', 'state', 'region', 'branch_id', 'payment_terms'
]

for col in text_cols:
    customers[col] = customers[col].str.title()


In [34]:
customers['customer_since'] = pd.to_datetime(customers['customer_since'], errors='coerce')
customers['last_purchase_date'] = pd.to_datetime(customers['last_purchase_date'], errors='coerce')

customers[['customer_since', 'last_purchase_date']].head()


,customer_since,last_purchase_date
0,2024-04-10,2024-04-25
1,2022-01-19,2024-12-04
2,2017-02-14,2019-06-11
3,2017-12-20,2021-12-26
4,2023-07-27,2024-02-17


In [35]:
customers['customer_tenure_years'] = (
    (pd.Timestamp.today() - customers['customer_since']).dt.days / 365
)


In [36]:
customers['days_since_last_purchase'] = (
    (pd.Timestamp.today() - customers['last_purchase_date']).dt.days
)


In [37]:
customers['credit_utilization_pct'] = (
    customers['current_balance'] / customers['credit_limit']
)
customers['avg_purchase_value'] = (
    customers['total_purchase_value'] / customers['customer_tenure_years'].replace(0, 1)
)


In [38]:
customers.drop_duplicates(inplace=True)


In [39]:
print("=== DUPLICATE CUSTOMER IDs ===")
print(customers['customer_id'].duplicated().sum())

print("\n=== REGION VALUES ===")
print(customers['region'].unique())

print("\n=== DATE CONVERSION CHECK ===")
print(customers[['customer_since', 'last_purchase_date']].head())

print("\n=== NUMERIC SUMMARY ===")
numeric_cols = [
    'credit_limit', 'current_balance', 'total_purchase_value', 'customer_rating'
]
print(customers[numeric_cols].describe())


=== DUPLICATE CUSTOMER IDs ===
0

=== REGION VALUES ===
<StringArray>
['East', 'South', 'West', 'North', 'Central', 'North East']
Length: 6, dtype: str

=== DATE CONVERSION CHECK ===
  customer_since last_purchase_date
0     2024-04-10         2024-04-25
1     2022-01-19         2024-12-04
2     2017-02-14         2019-06-11
3     2017-12-20         2021-12-26
4     2023-07-27         2024-02-17

=== NUMERIC SUMMARY ===
       credit_limit  current_balance  total_purchase_value  customer_rating
count  5.000000e+02     5.000000e+02          5.000000e+02        500.00000
mean   4.915253e+05     2.466805e+05          2.480429e+06          2.98200
std    3.720844e+05     2.588476e+05          1.454708e+06          1.40913
min    1.055200e+04     6.300000e+01          5.173000e+03          1.00000
25%    2.106442e+05     3.739025e+04          1.221794e+06          2.00000
50%    4.171450e+05     1.785700e+05          2.546906e+06          3.00000
75%    7.136942e+05     3.488998e+05        

In [40]:
customers.to_csv('../data/processed/customers.csv', index=False)
print("Processed customers.csv saved to data/processed/")


Processed customers.csv saved to data/processed/
